In [149]:
"""
SEC Filing Scraper
@Author: Nicholas Jung
"""

import requests
import pandas as pd

In [150]:
headers = {'User-Agent':
          "nicholasjung0503@gmail.com"}

target_tickers = ['LULU', 'UAA']

In [151]:
print("getting the CIK master list..")
tickers_url = "https://www.sec.gov/files/company_tickers.json"
ciks_dict = requests.get(
    tickers_url,
    headers = headers
)
df_ciks = pd.DataFrame.from_dict(ciks_dict.json(), orient= "index")
df_ciks['cik_str']= df_ciks['cik_str'].astype(str).str.zfill(10)

getting the CIK master list..


### Pulling the XBRL tags

In [152]:
metrics_to_pull = {
    'Net_Revenue': ['Revenues', 'RevenueFromContractWithCustomerExcludingAssessedTax', 'SalesRevenueNet'],
    'Cost_of_Revenue': ['CostOfGoodsAndServicesSold'],
    'Operating_Income': ['OperatingIncomeLoss']
}



### Little bit of logic
- since APIs are fragile, have to implement at try/except block that acts as a safety net. Otherwise, if a file is corrupted then the entire thing crashes, losing all your progress at the same time.


In [153]:
allCompanyData = []
for ticker in target_tickers:
    print(f"Processing {ticker}..")
    try:
        cik = df_ciks[df_ciks["ticker"] == ticker]['cik_str'].iloc[0]
        facts_url = f'https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json'
        companyFacts = requests.get( facts_url,
                                     headers = headers
                                   )
        ## saves raw dictionary
        usgaapCompany = companyFacts.json()['facts']['us-gaap']

        for metric_name, xbrl_tags in metrics_to_pull.items():
            print(f"Hunting for: {metric_name}...")

            valid_tag = None

            for tag in xbrl_tags:
                if tag in usgaapCompany:
                    valid_tag = tag
                    break
            if valid_tag:
                print(f"Success : we found {valid_tag}")
                raw_data = usgaapCompany[valid_tag]['units']['USD']

                df_metric = pd.DataFrame(raw_data)
                df_quarterly = df_metric[df_metric['form'] == "10-Q"].copy() ##
                
                if 'segment' in df_quarterly.columns:
                    df_quarterly = df_quarterly[df_quarterly['segment'].isna()]
                    
                df_quarterly = df_quarterly.reset_index(drop = True)
                ## slicing two columns, making sure we rename
                df_quarterly = df_quarterly[['end', 'val']].rename(columns = {'end': 'Date', 'val': 'Value'})
                df_quarterly['Ticker'] = ticker
                df_quarterly['Metric'] = metric_name

                                                                       
                allCompanyData.append(df_quarterly)

            else:
                print(f" Warning: Could not find SEC tag for {metric_name}")
        
    except Exception as e:
        print(f"Error processing {ticker}: {e}")

print(allCompanyData)

Processing LULU..
Hunting for: Net_Revenue...
Success : we found RevenueFromContractWithCustomerExcludingAssessedTax
Hunting for: Cost_of_Revenue...
Success : we found CostOfGoodsAndServicesSold
Hunting for: Operating_Income...
Success : we found OperatingIncomeLoss
Processing UAA..
Hunting for: Net_Revenue...
Success : we found Revenues
Hunting for: Cost_of_Revenue...
Success : we found CostOfGoodsAndServicesSold
Hunting for: Operating_Income...
Success : we found OperatingIncomeLoss
[          Date       Value Ticker       Metric
0   2018-10-28  2120861000   LULU  Net_Revenue
1   2018-10-28   747655000   LULU  Net_Revenue
2   2019-05-05   782315000   LULU  Net_Revenue
3   2019-08-04  1665667000   LULU  Net_Revenue
4   2019-08-04   883352000   LULU  Net_Revenue
..         ...         ...    ...          ...
59  2025-05-04  2370660000   LULU  Net_Revenue
60  2025-08-03  4895879000   LULU  Net_Revenue
61  2025-08-03  2525219000   LULU  Net_Revenue
62  2025-11-02  7461799000   LULU  Net_

### Submitting all into one, clean dataset.

In [184]:
pd.concat(allCompanyData)

,Date,Value,Ticker,Metric
0,2018-10-28,2120861000,LULU,Net_Revenue
1,2018-10-28,747655000,LULU,Net_Revenue
2,2019-05-05,782315000,LULU,Net_Revenue
3,2019-08-04,1665667000,LULU,Net_Revenue
4,2019-08-04,883352000,LULU,Net_Revenue
...,...,...,...,...
153,2025-06-30,3323000,UAA,Operating_Income
154,2025-09-30,20369000,UAA,Operating_Income
155,2025-09-30,17046000,UAA,Operating_Income
156,2025-12-31,-129411000,UAA,Operating_Income


In [186]:
df_final = pd.concat(allCompanyData)
df_final = df_final.reset_index()
# df_test_revenue = df_final[df_final['Metric'] == 'Net_Revenue']

### Changing the Date format, maybe will help with pivoting

In [187]:
df_final["Date"] = pd.to_datetime(df_final['Date'], format = 'mixed')
df_final['clean_date'] = df_final["Date"].dt.strftime('%b %d, %Y')

In [188]:
## Testing df_revenue metric since it shows NA once pivoted
df_final["Metric"].unique()
df_final[df_final["Date"] == "2009-08-02"]

,index,Date,Value,Ticker,Metric,clean_date
142,0,2009-08-02,24185000,LULU,Operating_Income,"Aug 02, 2009"
143,1,2009-08-02,14332000,LULU,Operating_Income,"Aug 02, 2009"


### Explanation for why there are no Cost_of_Revenue and Net_Revenue prior to 2017
- Accounting standard called ASC 606 was in effect, and changed how companies were required to recognize and report revenue.

In [198]:
df_pivot = df_final.pivot_table(index = ["Date", "Ticker"], 
                                     columns = ["Metric"], 
                                     values = "Value",
                                    aggfunc = "sum").reset_index()

### Changing to fill all NA values to 0.

In [216]:
df_pivot = df_pivot.fillna(0)
df_pivot.head()

Metric,Date,Ticker,Cost_of_Revenue,Net_Revenue,Operating_Income
0,2009-06-30,UAA,0.0,0.0,14658000.0
1,2009-08-02,LULU,0.0,0.0,38517000.0
2,2009-09-30,UAA,0.0,0.0,105403000.0
3,2009-11-01,LULU,0.0,0.0,66037000.0
4,2010-03-31,UAA,0.0,0.0,13584000.0


In [218]:
df_clean = df_pivot.sort_values(by= "Ticker")
df_clean.head()

Metric,Date,Ticker,Cost_of_Revenue,Net_Revenue,Operating_Income
33,2014-11-02,LULU,0.000000e+00,0.000000e+00,6.000020e+08
83,2023-04-30,LULU,1.699974e+09,4.001584e+09,8.028280e+08
65,2020-05-03,LULU,6.351200e+08,1.303924e+09,6.550200e+07
35,2015-05-03,LULU,0.000000e+00,0.000000e+00,1.360720e+08
81,2022-10-30,LULU,6.383992e+09,1.439114e+10,2.732818e+09


## Initialized new columns:
- Gross Profit
- Gross Margin PCT
- Operating Margin PCT


In [230]:
df_clean["Gross_Profit"] = df_clean["Net_Revenue"] - df_clean["Cost_of_Revenue"]
df_clean["Gross_Margin_Pct"] = df_clean["Gross_Profit"]/df_clean["Net_Revenue"]
df_clean["Operating_Margin_Pct"] = df_clean["Operating_Income"] / df_clean["Net_Revenue"]

In [231]:
df_clean.fillna(0)

Metric,Date,Ticker,Cost_of_Revenue,Net_Revenue,Operating_Income,Gross_Profit,Gross_Margin_Pct,Operating_Marging_Pct,Operating_Margin_Pct
33,2014-11-02,LULU,0.000000e+00,0.000000e+00,6.000020e+08,0.000000e+00,0.000000,inf,inf
83,2023-04-30,LULU,1.699974e+09,4.001584e+09,8.028280e+08,2.301610e+09,0.575175,0.200628,0.200628
65,2020-05-03,LULU,6.351200e+08,1.303924e+09,6.550200e+07,6.688040e+08,0.512916,0.050235,0.050235
35,2015-05-03,LULU,0.000000e+00,0.000000e+00,1.360720e+08,0.000000e+00,0.000000,inf,inf
81,2022-10-30,LULU,6.383992e+09,1.439114e+10,2.732818e+09,8.007146e+09,0.556394,0.189896,0.189896
...,...,...,...,...,...,...,...,...,...
58,2019-03-31,UAA,1.319870e+09,2.409444e+09,7.051800e+07,1.089574e+09,0.452210,0.029267,0.029267
60,2019-06-30,UAA,3.869502e+09,7.176360e+09,2.459000e+07,3.306858e+09,0.460799,0.003427,0.003427
62,2019-09-30,UAA,5.552918e+09,1.051073e+10,6.032340e+08,4.957808e+09,0.471690,0.057392,0.057392
32,2014-09-30,UAA,0.000000e+00,0.000000e+00,7.075240e+08,0.000000e+00,0.000000,inf,inf


SyntaxError: invalid syntax (4238376967.py, line 1)